# Chapter 19: Training and Deploying TensorFlow Models at Scale

Bab ini membahas bagaimana membawa model dari lingkungan pengembangan ke tahap produksi (deployment) serta teknik mempercepat pelatihan menggunakan infrastruktur skala besar.

---

## 19.1 TensorFlow Serving
TensorFlow Serving adalah sistem penyajian model yang efisien dan fleksibel, dirancang untuk lingkungan produksi.
* **SavedModel**: Model disimpan dalam format `SavedModel` yang berisi *MetaGraph* (grafik komputasi) dan *Function Signatures*.
* **Versi & Tag**: Mendukung manajemen versi model secara otomatis tanpa harus menghentikan server.
* **Skalabilitas**: Dapat diakses melalui **REST API** atau **gRPC**. Jika beban kerja meningkat, TF Serving dapat diluncurkan di banyak server untuk meningkatkan *Queries Per Second* (QPS).

---

## 19.2 Google Cloud Vertex AI (GCP)
Menjalankan model di cloud memberikan keuntungan berupa skalabilitas otomatis dan manajemen infrastruktur.
* **Service Prediksi**: Platform seperti Google Cloud Vertex AI memungkinkan kita menjalankan TF Serving dengan sistem keamanan (enkripsi dan autentikasi) yang sudah terintegrasi.
* **Batch Prediction**: Mendukung prediksi dalam jumlah besar sekaligus secara terjadwal.

---

## 19.3 TensorFlow Lite (TFLite)
Untuk menjalankan model di perangkat mobile (Android/iOS) atau perangkat *embedded* (Raspberry Pi, Edge TPU), model harus dioptimalkan.
* **Efisiensi**: TFLite melakukan kuantisasi (mengubah bobot float 32-bit ke int 8-bit) untuk memperkecil ukuran model dan mempercepat eksekusi tanpa banyak mengorbankan akurasi.
* **Interpreter**: Menggunakan *interpreter* khusus yang ringan untuk menjalankan model di perangkat dengan batasan memori dan daya.

---

## 19.4 TensorFlow.js
Memungkinkan eksekusi model langsung di dalam *browser* menggunakan JavaScript.
* **Keuntungan**: Privasi data terjaga (data tidak dikirim ke server), respon model sangat cepat karena diproses di sisi klien, dan dapat memanfaatkan GPU pengguna via WebGL.

---

## 19.5 Strategi Penggunaan GPU
Melatih Neural Network besar pada CPU sangat lambat. Penggunaan GPU (atau banyak GPU) adalah standar industri.
* **Manajemen RAM GPU**: Secara *default*, TensorFlow akan mengambil seluruh RAM GPU yang tersedia saat pertama kali dijalankan. Hal ini bisa diatur menggunakan `tf.config.experimental.set_memory_growth` agar TF hanya mengambil memori sesuai kebutuhan.
* **Dynamic Placer**: Algoritma yang mendistribusikan operasi ke berbagai perangkat berdasarkan estimasi waktu komputasi dan ketersediaan memori.

---

## 19.6 Model Parallelism vs Data Parallelism
Saat melatih model yang sangat besar, kita harus membagi beban kerja ke banyak device.

1.  **Model Parallelism**: Membagi bagian-bagian model (misal: layer 1-5 di GPU 0, layer 6-10 di GPU 1). Teknik ini sulit karena layer atas seringkali harus menunggu hasil dari layer bawah.
2.  **Data Parallelism**: Strategi paling populer. Model direplikasi ke semua GPU yang tersedia, namun setiap replika melatih **mini-batch** data yang berbeda secara bersamaan.



---

## 19.7 Sinkronisasi Gradien (Data Parallelism)
Ada dua strategi utama untuk memperbarui parameter model saat menggunakan Data Parallelism:
* **Synchronous Updates**: Parameter server/aggregator menunggu gradien dari **semua** replika selesai dihitung sebelum memperbarui bobot. Ini lebih stabil tetapi kecepatan bergantung pada GPU yang paling lambat.
* **Asynchronous Updates**: Parameter langsung diperbarui setiap kali ada satu replika yang selesai menghitung gradien. Ini lebih cepat namun bisa menyebabkan masalah "stale gradients" (gradien usang).



---

## 19.8 Distribusi Strategi di TensorFlow
TensorFlow menyediakan API `tf.distribute.Strategy` untuk mempermudah pelatihan terdistribusi:
* **MirroredStrategy**: Melakukan replikasi model pada semua GPU di satu mesin (Sinkron).
* **MultiWorkerMirroredStrategy**: Mirip MirroredStrategy tetapi untuk banyak mesin/server.
* **ParameterServerStrategy**: Memisahkan tugas antara mesin yang menyimpan parameter dan mesin yang melakukan komputasi (Worker).